In [1]:
import random
import numpy as np
import pandas as pd

random.seed(5)
np.random.seed(5)

products = pd.DataFrame({
    "product_id": [f"P{i:02d}" for i in range(1, 13)],
    "name": ["Headphones", "Phone Case", "Smart Speaker", "Desk Lamp", "Cookware Set", "Vacuum Bottle",
             "T-Shirt", "Jacket", "Sneakers", "Yoga Mat", "Dumbbell Set", "Fitness Band"],
    "category": ["Electronics"] * 3 + ["Home"] * 3 + ["Clothing"] * 3 + ["Sports"] * 3,
    "price": [79.0, 15.0, 59.0, 35.0, 90.0, 25.0, 20.0, 85.0, 70.0, 30.0, 60.0, 45.0],
    "cost": [48.0, 5.0, 38.0, 29.0, 76.0, 20.0, 7.0, 40.0, 33.0, 12.0, 35.0, 24.0],
})

cities = ["Berlin", "Rome", "Madrid", "Paris", "Vienna", "Warsaw"]
customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(1, 401)],
    "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 1100, 400), unit="D"),
    "city": [random.choice(cities) for _ in range(400)],
    "channel": [random.choice(["Organic", "Ads", "Referral"]) for _ in range(400)],
})
customers["signup_date"] = customers["signup_date"].dt.strftime("%Y-%m-%d")

weights = np.random.lognormal(0, 1, 400)
weights = weights / weights.sum()
pop = np.array([3, 4, 2, 3, 1, 3, 4, 2, 3, 3, 2, 2], dtype=float)
pop = pop / pop.sum()

rows = []
oid = 1000
for d in pd.date_range("2025-01-01", "2025-12-31"):
    lam = 14 * (1.8 if d.month in (11, 12) else 1.0) * (1.25 if d.dayofweek >= 5 else 1.0)
    for _ in range(np.random.poisson(lam)):
        oid += 1
        c = np.random.choice(customers["customer_id"], p=weights)
        p = np.random.choice(products["product_id"], p=pop)
        cat = products.loc[products["product_id"] == p, "category"].iloc[0]
        qty = int(np.random.choice([1, 2, 3, 4], p=[0.6, 0.25, 0.1, 0.05]))
        if cat == "Home" and d.month >= 7:
            disc = random.choice([0.2, 0.25, 0.3])
        else:
            disc = random.choice([0, 0, 0, 0.05, 0.1])
        r_ret = 0.15 if cat == "Clothing" else 0.04
        u = random.random()
        status = "Returned" if u < r_ret else ("Cancelled" if u < r_ret + 0.04 else "Delivered")
        pay = random.choice(["Card", "Card", "PayPal", "Transfer"])
        rows.append([f"O{oid}", d.strftime("%Y-%m-%d"), c, p, qty, disc, status, pay])

orders = pd.DataFrame(rows, columns=["order_id", "order_date", "customer_id", "product_id",
                                     "quantity", "discount_pct", "status", "payment_method"])
n = len(orders)
orders["discount_pct"] = orders["discount_pct"].astype(object)
for i in random.sample(range(n), 70):
    orders.loc[i, "order_date"] = pd.to_datetime(orders.loc[i, "order_date"]).strftime("%d.%m.%Y")
for i in random.sample(range(n), 90):
    s = orders.loc[i, "status"]
    orders.loc[i, "status"] = random.choice([s.lower(), s.upper(), " " + s])
for i in random.sample(range(n), 120):
    orders.loc[i, "discount_pct"] = f"{int(float(orders.loc[i, 'discount_pct']) * 100)}%"
for i in random.sample(range(n), 12):
    orders.loc[i, "quantity"] = -1
for i in random.sample(range(n), 45):
    orders.loc[i, "customer_id"] = None
for i in random.sample(range(n), 15):
    orders.loc[i, "product_id"] = "P99"
orders = pd.concat([orders, orders.sample(25, random_state=2)]).sample(frac=1, random_state=9).reset_index(drop=True)

for i in random.sample(range(400), 40):
    customers.loc[i, "city"] = random.choice([customers.loc[i, "city"].upper(), " " + customers.loc[i, "city"].lower()])
products["price"] = products["price"].astype(object)
products.loc[products["product_id"] == "P05", "price"] = "90,00"

products.to_csv("products.csv", index=False)
customers.to_csv("customers.csv", index=False)
orders.to_csv("orders.csv", index=False)